# 第4.2步：批量检索生成 RAG 数据集

基于前一步的向量库与目标城市样本（默认汉中 2024），批量检索 TopK，按季节划分 train/val/test，并输出：
- `rag_results_all.npz` / `rag_results_all.json`
- 拆分的 `rag_results_all_train/val/test.npz`

## 1. 参数
按需修改输入输出路径、TopK、批大小等。

In [1]:
from pathlib import Path

INPUT_LEN = 168
TOPK = 3
BATCH_SIZE = 256
POOLING = 'last'
CHRONOS_MODEL_PATH = 'chronos'

# 目录配置（默认使用目标城市样本作为查询，查询城市向量库作为检索库）
VECTOR_DB_DIR = Path('./shaanxi_cities_temperature/vector_db_w192_s6/query')
SAMPLES_DIR   = Path('./shaanxi_cities_temperature/samples_w192_s6/query')
QUERY_NPZ     = Path('./shaanxi_cities_temperature/samples_w192_s6/target/汉中_samples_w192_s6.npz')
SAVE_NPZ      = Path('./shaanxi_cities_temperature/rag_results_all.npz')
SAVE_JSON     = Path('./shaanxi_cities_temperature/rag_results_all.json')

print('VECTOR_DB_DIR =', VECTOR_DB_DIR)
print('SAMPLES_DIR   =', SAMPLES_DIR)
print('QUERY_NPZ     =', QUERY_NPZ)
print('SAVE_NPZ      =', SAVE_NPZ)
print('SAVE_JSON     =', SAVE_JSON)

VECTOR_DB_DIR = shaanxi_cities_temperature/vector_db_w192_s6/query
SAMPLES_DIR   = shaanxi_cities_temperature/samples_w192_s6/query
QUERY_NPZ     = shaanxi_cities_temperature/samples_w192_s6/target/汉中_samples_w192_s6.npz
SAVE_NPZ      = shaanxi_cities_temperature/rag_results_all.npz
SAVE_JSON     = shaanxi_cities_temperature/rag_results_all.json


## 2. 工具与批量检索逻辑

In [2]:
import json
import pickle
from typing import Dict, List, Optional
import numpy as np
import torch
import faiss
from chronos import ChronosPipeline
from tqdm import tqdm

CITY_SLUG = {
    "西安": "xian", "郑州": "zhengzhou", "太原": "taiyuan", "石家庄": "shijiazhuang",
    "济南": "jinan", "呼和浩特": "huhehaote", "银川": "yinchuan", "兰州": "lanzhou",
    "西宁": "xining", "大同": "datong", "汉中": "hanzhong",
}

def safe_slug(name: str) -> str:
    if name in CITY_SLUG:
        return CITY_SLUG[name]
    cleaned = ''.join(ch if ch.isascii() and ch.isalnum() else '_' for ch in name).strip('_')
    return cleaned or 'city'

def ts_to_str(ts) -> str:
    ts_hour = np.datetime64(ts, 'h')
    return np.datetime_as_string(ts_hour, unit='h').replace('-', '').replace('T', '')

def month_to_season(m: int) -> str:
    if m in (12,1,2): return 'winter'
    if m in (3,4,5): return 'spring'
    if m in (6,7,8): return 'summer'
    if m in (9,10,11): return 'autumn'
    return 'unknown'

class CityTempRAG:
    def __init__(self, vector_db_dir: Path, samples_dir: Path, chronos_model_path: str, pooling: str = 'last', device: Optional[str] = None):
        self.vector_db_dir = Path(vector_db_dir)
        self.samples_dir = Path(samples_dir)
        self.device = device or ('cuda' if torch.cuda.is_available() else 'cpu')
        self.pooling = pooling
        self.chronos_model_path = chronos_model_path
        self._load_vector_db()
        self._init_chronos()
        self.npz_cache: Dict[str, Dict[str, np.ndarray]] = {}

    def _load_vector_db(self):
        self.embeddings = np.load(self.vector_db_dir / 'embeddings.npy')
        with open(self.vector_db_dir / 'id_mapping.pkl', 'rb') as f:
            id_map = pickle.load(f)
        self.sample_id_to_idx = id_map['sample_id_to_idx']
        self.idx_to_sample_id = id_map['idx_to_sample_id']
        with open(self.vector_db_dir / 'meta.json', 'r') as f:
            self.meta: List[Dict] = json.load(f)
        self.meta_map = {m['sample_id']: m for m in self.meta}

        index_path = self.vector_db_dir / 'faiss_index.bin'
        if index_path.exists():
            cpu_index = faiss.read_index(str(index_path))
        else:
            dim = self.embeddings.shape[1]
            cpu_index = faiss.IndexFlatIP(dim)
            cpu_index.add(self.embeddings.astype(np.float32))
        self.search_index = cpu_index
        if faiss.get_num_gpus() > 0:
            res = faiss.StandardGpuResources()
            self.search_index = faiss.index_cpu_to_gpu(res, 0, cpu_index)
            print('Using FAISS GPU index')
        else:
            print('Using FAISS CPU index')
        self.emb_normalized = abs(np.linalg.norm(self.embeddings[0]) - 1.0) < 1e-3
        print(f"Loaded vector DB: {len(self.embeddings)} vectors, dim={self.embeddings.shape[1]}")

    def _init_chronos(self):
        print(f'Loading Chronos on {self.device} ...')
        self.chronos = ChronosPipeline.from_pretrained(
            self.chronos_model_path,
            device_map=self.device,
            dtype=torch.float32,
        )
        print('Chronos loaded')

    def _pool(self, embeds: torch.Tensor) -> torch.Tensor:
        if self.pooling == 'mean': return embeds.mean(dim=1)
        if self.pooling == 'last': return embeds[:, -1, :]
        if self.pooling == 'max': return embeds.max(dim=1)[0]
        if self.pooling == 'weighted_mean':
            seq_len = embeds.shape[1]
            w = torch.arange(1, seq_len+1, device=embeds.device, dtype=torch.float32)
            w = w / w.sum()
            return (embeds * w.view(1,-1,1)).sum(dim=1)
        raise ValueError('Unknown pooling')

    def embed_batch(self, batch_series: np.ndarray) -> np.ndarray:
        ctx = torch.tensor(batch_series[:, -INPUT_LEN:], dtype=torch.float32)
        with torch.no_grad():
            embeds_raw, _ = self.chronos.embed(ctx)
            pooled = self._pool(embeds_raw)
            q = pooled.cpu().numpy()
        if self.emb_normalized:
            q = q / (np.linalg.norm(q, axis=1, keepdims=True) + 1e-8)
        return q.astype(np.float32)

    def batch_search(self, batch_series: np.ndarray, topk: int):
        queries = self.embed_batch(batch_series)
        return self.search_index.search(queries, topk)

    def fetch_sample(self, sample_id: str):
        meta = self.meta_map[sample_id]
        npz_path = Path(meta['npz_path'])
        if not npz_path.is_absolute():
            npz_path = self.samples_dir / npz_path.name
        npz_path = str(npz_path)
        if npz_path not in self.npz_cache:
            self.npz_cache[npz_path] = dict(np.load(npz_path))
        data = self.npz_cache[npz_path]
        idx = meta['idx_in_city']
        return data['input'][idx], data['target'][idx]

/usr/local/miniconda3/envs/timerag/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 3. 运行批量检索与划分
- 对 `QUERY_NPZ` 中的全部样本批量检索 TopK。
- 按季节划分 train/val/test（12/1/2 冬，3/4/5 春，6/7/8 夏，9/10/11 秋；各季 0.8/0.1/0.1 时序切分）。
- 保存 `all` 及 `train/val/test` npz。

In [3]:
import numpy as np

data = np.load(QUERY_NPZ)
inputs = data['input']
targets = data['target']
start_times = data['start_time']
assert len(inputs) == len(targets) == len(start_times)
print('query samples:', len(inputs))

city_name = QUERY_NPZ.stem.replace('_samples_w192_s6','')
city_slug = safe_slug(city_name)

rag = CityTempRAG(
    vector_db_dir=VECTOR_DB_DIR,
    samples_dir=SAMPLES_DIR,
    pooling=POOLING,
    chronos_model_path=CHRONOS_MODEL_PATH,
)

all_query_ids = []
all_seasons = []
all_splits = []
all_start_times = []
all_top_ids = []
all_top_scores = []
all_query_inputs = []
all_query_targets = []
all_top_inputs = []
all_top_targets = []

bs = BATCH_SIZE
for start in tqdm(range(0, len(inputs), bs), desc='processing batches'):
    end = min(start + bs, len(inputs))
    batch_inputs = inputs[start:end]
    batch_targets = targets[start:end]
    batch_times = start_times[start:end]
    scores, idxs = rag.batch_search(batch_inputs, TOPK)
    for local_i in range(len(batch_inputs)):
        q_inp = batch_inputs[local_i]
        q_tgt = batch_targets[local_i]
        ts = batch_times[local_i]
        ts_str = ts_to_str(ts)
        month = int(str(np.datetime64(ts, 'M'))[-2:])
        season = month_to_season(month)
        qid = f"{city_slug}_{ts_str}"

        top_ids = []
        top_scores = []
        top_inp_list = []
        top_tgt_list = []
        for score, hit_idx in zip(scores[local_i], idxs[local_i]):
            sid = rag.idx_to_sample_id[int(hit_idx)]
            inp_r, tgt_r = rag.fetch_sample(sid)
            top_ids.append(sid)
            top_scores.append(float(score))
            top_inp_list.append(inp_r)
            top_tgt_list.append(tgt_r)

        all_query_ids.append(qid)
        all_seasons.append(season)
        all_splits.append('unassigned')
        all_start_times.append(str(ts))
        all_query_inputs.append(q_inp)
        all_query_targets.append(q_tgt)
        all_top_ids.append(top_ids)
        all_top_scores.append(top_scores)
        all_top_inputs.append(top_inp_list)
        all_top_targets.append(top_tgt_list)

# 按季节划分 train/val/test
season_to_indices = {}
for i, season in enumerate(all_seasons):
    season_to_indices.setdefault(season, []).append(i)
for season, idx_list in season_to_indices.items():
    idx_sorted = sorted(idx_list, key=lambda k: all_start_times[k])
    n = len(idx_sorted)
    n_train = int(n*0.8)
    n_val = int(n*0.1)
    n_test = n - n_train - n_val
    for j, k in enumerate(idx_sorted):
        if j < n_train:
            all_splits[k] = 'train'
        elif j < n_train + n_val:
            all_splits[k] = 'val'
        else:
            all_splits[k] = 'test'
    print(f"{season}: total {n}, train {n_train}, val {n_val}, test {n_test}")

query samples: 1433
Using FAISS CPU index
Loaded vector DB: 481810 vectors, dim=1024
Loading Chronos on cuda ...
Chronos loaded


processing batches: 100%|██████████| 6/6 [01:19<00:00, 13.17s/it]

winter: total 333, train 266, val 33, test 34
spring: total 368, train 294, val 36, test 38
summer: total 368, train 294, val 36, test 38
autumn: total 364, train 291, val 36, test 37


## 4. 保存 all/train/val/test 数据

In [4]:
import json

def save_split(split_name: str):
    mask = np.array(all_splits) == split_name
    out = SAVE_NPZ.with_name(SAVE_NPZ.stem + f'_{split_name}.npz')
    np.savez_compressed(
        out,
        query_ids=np.array(all_query_ids)[mask],
        start_times=np.array(all_start_times)[mask],
        seasons=np.array(all_seasons)[mask],
        splits=np.array(all_splits)[mask],
        query_inputs=np.array(all_query_inputs, dtype=np.float32)[mask],
        query_targets=np.array(all_query_targets, dtype=np.float32)[mask],
        top_ids=np.array(all_top_ids, dtype=object)[mask],
        top_scores=np.array(all_top_scores, dtype=np.float32)[mask],
        top_inputs=np.array(all_top_inputs, dtype=np.float32)[mask],
        top_targets=np.array(all_top_targets, dtype=np.float32)[mask],
    )
    print('saved', split_name, '->', out)

# all
SAVE_NPZ.parent.mkdir(parents=True, exist_ok=True)
np.savez_compressed(
    SAVE_NPZ,
    query_ids=np.array(all_query_ids),
    start_times=np.array(all_start_times),
    seasons=np.array(all_seasons),
    splits=np.array(all_splits),
    query_inputs=np.array(all_query_inputs, dtype=np.float32),
    query_targets=np.array(all_query_targets, dtype=np.float32),
    top_ids=np.array(all_top_ids, dtype=object),
    top_scores=np.array(all_top_scores, dtype=np.float32),
    top_inputs=np.array(all_top_inputs, dtype=np.float32),
    top_targets=np.array(all_top_targets, dtype=np.float32),
)
print('saved all ->', SAVE_NPZ)

# json meta
records = []
for i in range(len(all_query_ids)):
    records.append({
        'query_id': all_query_ids[i],
        'start_time': all_start_times[i],
        'season': all_seasons[i],
        'split': all_splits[i],
        'top_ids': all_top_ids[i],
        'top_scores': all_top_scores[i],
    })
SAVE_JSON.parent.mkdir(parents=True, exist_ok=True)
with open(SAVE_JSON, 'w') as f:
    json.dump(records, f, ensure_ascii=False, indent=2)
print('saved json ->', SAVE_JSON)

for split in ['train','val','test']:
    save_split(split)

saved all -> shaanxi_cities_temperature/rag_results_all.npz
saved json -> shaanxi_cities_temperature/rag_results_all.json
saved train -> shaanxi_cities_temperature/rag_results_all_train.npz
saved val -> shaanxi_cities_temperature/rag_results_all_val.npz
saved test -> shaanxi_cities_temperature/rag_results_all_test.npz


## 5. 查看输出文件

In [5]:
import glob
print('NPZ/JSON 输出：')
for f in sorted(glob.glob(str(SAVE_NPZ.parent / 'rag_results_all*.npz')) + [str(SAVE_JSON)]):
    print(f)

NPZ/JSON 输出：
shaanxi_cities_temperature/rag_results_all.json
shaanxi_cities_temperature/rag_results_all.npz
shaanxi_cities_temperature/rag_results_all_test.npz
shaanxi_cities_temperature/rag_results_all_train.npz
shaanxi_cities_temperature/rag_results_all_val.npz
